In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

df = pd.read_csv('historical_data.csv')
df = pd.DataFrame(df)

# Skriv ut de första 5 raderna för att dubbelkolla att det ser bra ut
df.head()

,id,day,event_type,category,region,device,account_age_days,num_prev_listings,prev_reports_30d,verification_level,price,num_images,message_length,contains_off_platform,urgency_words,payment_attempt,time_to_first_response_min,is_suspicious
0,0,8,ad_post,other,urban,android,38.4,2,1,1,594.16,3,91,0,1,0,2.3,0
1,1,4,ad_post,fashion,urban,android,20.0,1,0,1,134.47,2,150,0,0,0,13.6,0
2,2,4,ad_post,other,metro,ios,46.7,3,1,1,198.52,3,72,0,0,0,4.2,0
3,3,3,ad_post,furniture,metro,android,44.3,3,1,2,141.20,3,0,0,0,0,19.8,0
4,4,1,ad_post,electronics,metro,web,211.2,5,0,0,81.39,3,9,0,0,1,23.3,0


## Modellering och Jämförelse

In [ ]:
# Delar upp i X (features) och y (target)
target = 'is_suspicious'
X = df.drop(columns=[target])
y = df[target].astype(int)

# Tar bort ID i columnen
if 'id' in X.columns:
    X = X.drop(columns=['id'])

# Hittar vilka kolumner som är numeriska och vilka som är kategorier och text
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]


# Gör en pipeline där preprocessing används för alla modeller
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
    ]
)

# Moddeller / Jämföra
models = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200, class_weight='balanced')
}

# Corss validation, Stratified ser till att varje fold får nästan samma andel som en hel data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Metrics jämföra
# Precision - kolla hur många av de flaggade som verkligen var misstänkta
# Recall = koll hur storr andel av alla misstänkta man hittar
# f1 = balans mellan precision oc rreccall

scoring = {
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0),
}

# Körs cv för varje modell och sparar resultat
rows = []
for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocess', preprocess),
        ('model', model)
    ])

    cv_result = cross_validate(pipe, X, y, cv=cv, scoring=scoring)

    rows.append({
        'Model': name,
        'precision': cv_result['test_precision'].mean(),
        'recall': cv_result['test_recall'].mean(),
        'f1': cv_result['test_f1'].mean(),
    })

results = pd.DataFrame(rows).sort_values('f1', ascending=False)

print('\nFeatures:')
print('Numeriska: ', len(num_cols), num_cols[:5])
print('Kategoriska: ', len(cat_cols), cat_cols)

# Bästa modellen enligt Recall
best_recall = results.sort_values('recall', ascending=False).iloc[0]
# Bästa modell enligt F1, balans mellan precision och recall
best_f1 = results.sort_values('f1', ascending=False).iloc[0]

print('\n-- Bäst enligt Recall --')
print(
    'Model:', best_recall['Model'],
    'Recall:', round(best_recall['recall'], 3),
    'Precision:', round(best_recall['precision'], 3),
    'F1:', round(best_recall['f1'], 3)
)

print('\n-- Bäst enligt F1 --')
print(
    'Model:', best_f1['Model'],
    'F1:', round(best_f1['f1'], 3),
    'Recall:', round(best_f1['recall'], 3),
    'Precision:', round(best_f1['precision'], 3)
)

print('\n-- Resultat --')
print(results)



Features:
Numeriska:  12 ['day', 'account_age_days', 'num_prev_listings', 'prev_reports_30d', 'verification_level']
Kategoriska:  4 ['event_type', 'category', 'region', 'device']

-- Bäst enligt Recall --
Model: Logistic Regression Recall: 0.632 Precision: 0.192 F1: 0.294

-- Bäst enligt F1 --
Model: Logistic Regression F1: 0.294 Recall: 0.632 Precision: 0.192

-- Resultat --
                 Model  precision    recall        f1
1  Logistic Regression   0.191545  0.632385  0.293968
2        Random Forest   0.453333  0.007357  0.014445
0             Baseline   0.000000  0.000000  0.000000
